# Task 1: Image Feature Extraction
Extract image features from the Flickr8k dataset using VGG16 and save them to `features.pkl`.

In [1]:
!pip install --upgrade kaggle tensorflow pillow tqdm matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 54.5 MB/s  0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.8
    Uninstalling matplotlib-3.10.8:
      Successfully uninstalled matplotlib-3.10.8

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model


I0000 00:00:1777961941.356811  340097 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777961941.425702  340097 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777961943.500488  340097 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Step 1: Download Flickr8k Dataset
Make sure `kaggle.json` exists at `C:\Users\<you>\.kaggle\kaggle.json` with your Kaggle credentials before running this cell. The dataset is only downloaded once, subsequent runs will skip this step.

In [3]:
# Only download if the Images folder doesn't already exist
# kaggle.json must exist at C:\Users\<you>\.kaggle\kaggle.json
if os.path.exists("./flickr8k/Images"):
    print("Flickr8k already downloaded — skipping download.")
else:
    !kaggle datasets download -d adityajn105/flickr8k --unzip -p ./flickr8k
    print("Download complete!")

Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100%|██████████████████████████████████████| 1.04G/1.04G [00:29<00:00, 38.0MB/s]

Download complete!


In [4]:
images_dir = "./flickr8k/Images"
captions_file = "./flickr8k/captions.txt"

image_files = [f for f in os.listdir(images_dir) if f.endswith(".jpg")]
print(f"Total images found: {len(image_files)}")
print(f"Sample filenames: {image_files[:5]}")

Total images found: 8091
Sample filenames: ['3125628091_25a31709df.jpg', '2862931640_2501bd36c5.jpg', '3651476768_2bae721a6b.jpg', '3171035252_dba286ae5c.jpg', '2065309381_705b774f51.jpg']


## Step 2: Load VGG16 Model
VGG16 is pre-trained on ImageNet. We load the **full** model (`include_top=True`) and tap the second fully-connected layer (`fc2`), giving a **4096-dim semantic feature vector** per image. This is the standard image-captioning feature: high-level "what's in the image" without spatial bloat.

In [5]:
# Load full VGG16 with classifier head, then extract just the fc2 layer (4096-dim)
full_model = VGG16(weights='imagenet')
base_model = Model(inputs=full_model.input, outputs=full_model.get_layer('fc2').output)
base_model.trainable = False
print("VGG16 (fc2 output) loaded successfully!")
print(f"Output shape: {base_model.output_shape}")


I0000 00:00:1777961981.372053  340097 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7722 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:1f:00.0, compute capability: 7.5


553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step
VGG16 (fc2 output) loaded successfully!
Output shape: (None, 4096)


## Step 3: Extract Features
Each image is resized to 224×224 and preprocessed for VGG16. Images are processed in batches for efficiency. The fc2 layer produces a **4096-dim vector** per image — already 1D, no flattening needed.

In [6]:
BATCH_SIZE = 32

def extract_features_batch(image_files, images_dir, model, batch_size=BATCH_SIZE):
    features = {}
    for i in tqdm(range(0, len(image_files), batch_size), desc="Extracting features"):
        batch_names = image_files[i:i + batch_size]
        batch_arrays = []
        valid_names = []
        for img_name in batch_names:
            try:
                img = Image.open(os.path.join(images_dir, img_name)).convert("RGB")
                img = img.resize((224, 224))
                img_array = preprocess_input(np.array(img, dtype=np.float32))
                batch_arrays.append(img_array)
                valid_names.append(img_name)
            except Exception as e:
                print(f"Skipping {img_name}: {e}")
        batch_preds = model.predict(np.stack(batch_arrays), verbose=0)
        for name, feat in zip(valid_names, batch_preds):
            features[name] = feat  # Shape: (4096,) — fc2 output, already 1D
    return features


In [7]:
if os.path.exists("features.pkl"):
    print("features.pkl already exists — skipping extraction.")
    print("Delete features.pkl and re-run if you want to regenerate.")
else:
    features = extract_features_batch(image_files, images_dir, base_model)
    print(f"\nFeature extraction complete!")
    print(f"Total images processed: {len(features)}")
    print(f"Feature vector shape: {features[image_files[0]].shape}")

Extracting features:   0%|          | 0/253 [00:00<?, ?it/s]I0000 00:00:1777962001.209602  340448 service.cc:153] XLA service 0x7f2ae0030db0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777962001.209653  340448 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.3.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1777962001.287394  340448 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777962001.446854  340448 cuda_dnn.cc:461] Loaded cuDNN version 91900
E0000 00:00:1777962003.214917  340448 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1777962003.768215  340448 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmu


Feature extraction complete!
Total images processed: 8091
Feature vector shape: (4096,)


## Step 4: Save Features
Save the features dictionary to `features.pkl`. This file will be loaded by Task 3 for model training.

In [8]:
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

print("Features saved to 'features.pkl'")

Features saved to 'features.pkl'
